# CHECKPOINT 4.3: QUẢN LÝ GIÁO VIÊN & PHÂN CÔNG MÔN/LỚP

Notebook này phân tích và kiểm thử các quy tắc nghiệp vụ cốt lõi về Giáo viên và Phân công giảng dạy trong hệ thống **SmartEdu**:
1. **Quy mô giáo viên:** 15 giáo viên cơ hữu chia đều cho 5 môn học cốt lõi (3 giáo viên/môn).
2. **Quy tắc Bất biến 1 Giáo viên → 1 Môn học:** Giáo viên chỉ phụ trách đúng 1 môn trong 5 môn: Toán học, Ngữ văn, Tiếng Anh, Vật lý, Hóa học.
3. **Quy tắc 1 Giáo viên → Nhiều Lớp học:** 1 giáo viên có thể được phân công dạy môn chuyên trách của mình tại nhiều lớp học.
4. **Logical Unique Key & Chống trùng lặp:** `teacherId` + `classId` + `subjectId` + `academicYear`.
5. **Source of Truth & Bảo toàn Lịch sử:** Phân công lưu trữ tại `teacherAssignments`, gỡ phân công bằng chuyển trạng thái `INACTIVE`, không xóa vật lý.

In [ ]:
import json
from datetime import datetime

STANDARD_SUBJECTS = [
    {"id": "toan", "code": "TOAN", "name": "Toán học", "dept": "Tổ Tự Nhiên"},
    {"id": "van", "code": "NGU_VAN", "name": "Ngữ văn", "dept": "Tổ Xã Hội"},
    {"id": "anh", "code": "TIENG_ANH", "name": "Tiếng Anh", "dept": "Tổ Ngoại Ngữ"},
    {"id": "ly", "code": "VAT_LY", "name": "Vật lý", "dept": "Tổ Tự Nhiên"},
    {"id": "hoa", "code": "HOA_HOC", "name": "Hóa học", "dept": "Tổ Tự Nhiên"}
]

print(f"✓ Danh mục 5 môn học chuẩn: {[s['name'] for s in STANDARD_SUBJECTS]}")

## 1. Kiểm thử Ràng buộc: 1 Giáo viên → 1 Môn học duy nhất

In [ ]:
def validate_teacher_subject(subject_id):
    allowed_ids = [s["id"] for s in STANDARD_SUBJECTS]
    if not subject_id or subject_id.lower() not in allowed_ids:
        return False, f"Môn học '{subject_id}' không hợp lệ. Chỉ chấp nhận 5 môn: Toán, Văn, Anh, Lý, Hóa."
    return True, "Hợp lệ"

# Kiểm thử các trường hợp hợp lệ & bất hợp lệ
assert validate_teacher_subject("toan")[0] == True
assert validate_teacher_subject("van")[0] == True
assert validate_teacher_subject("anh")[0] == True
assert validate_teacher_subject("ly")[0] == True
assert validate_teacher_subject("hoa")[0] == True

# Các môn ngoài danh mục phải bị chặn
assert validate_teacher_subject("sinh")[0] == False
assert validate_teacher_subject("su")[0] == False
assert validate_teacher_subject("dia")[0] == False
print("✓ PASS: Ràng buộc 5 môn học chuẩn được bảo vệ tuyệt đối!")

## 2. Kiểm thử Logic Phân công: 1 Giáo viên dạy Nhiều Lớp & Khớp Môn

In [ ]:
def validate_assignment_eligibility(teacher, target_class, assignment_subject, academic_year, existing_assignments):
    # 1. Trạng thái giáo viên
    if teacher.get("status") != "ACTIVE":
        return False, f"Giáo viên {teacher['name']} không ở trạng thái ACTIVE."
    
    # 2. Trạng thái lớp
    if target_class.get("status") not in ["ACTIVE", "Đang hoạt động"]:
        return False, f"Lớp {target_class['name']} không ở trạng thái hoạt động."
    
    # 3. Ràng buộc khớp môn (1 Giáo viên chỉ dạy môn chuyên trách)
    if teacher.get("subjectId") != assignment_subject:
        return False, f"Không hợp lệ: Giáo viên chuyên trách môn {teacher.get('subjectId')}, không được gán môn {assignment_subject}."
    
    # 4. Trùng lặp phân công (Logical Unique Key)
    is_duplicate = any(
        a["teacherId"] == teacher["id"] and 
        a["classId"] == target_class["id"] and 
        a["subjectId"] == assignment_subject and 
        a["academicYear"] == academic_year and 
        a["status"] == "ACTIVE"
        for a in existing_assignments
    )
    if is_duplicate:
        return False, f"Giáo viên đã được phân công dạy môn {assignment_subject} tại lớp {target_class['name']}."
    
    return True, "Phân công hợp lệ"

# Mock dữ liệu
teacher_toan = {"id": "TCH-2026-001", "name": "Trần Quốc Việt", "subjectId": "toan", "status": "ACTIVE"}
teacher_van = {"id": "TCH-2026-004", "name": "Nguyễn Thu Hà", "subjectId": "van", "status": "ACTIVE"}
class_6A1 = {"id": "class_6A1", "name": "Lớp 6A1", "status": "Đang hoạt động"}
class_7A1 = {"id": "class_7A1", "name": "Lớp 7A1", "status": "Đang hoạt động"}

current_assignments = [
    {"teacherId": "TCH-2026-001", "classId": "class_6A1", "subjectId": "toan", "academicYear": "2026-2027", "status": "ACTIVE"}
]

# 1. Gán thầy Việt dạy Toán thêm ở lớp 7A1 -> Thành công (1 GV -> Nhiều lớp)
ok, msg = validate_assignment_eligibility(teacher_toan, class_7A1, "toan", "2026-2027", current_assignments)
assert ok == True

# 2. Gán thầy Việt (Toán) đi dạy Văn ở 7A1 -> Bị từ chối (Sai chuyên môn)
ok, msg = validate_assignment_eligibility(teacher_toan, class_7A1, "van", "2026-2027", current_assignments)
assert ok == False

# 3. Gán trùng thầy Việt dạy Toán ở 6A1 trong cùng năm học -> Bị từ chối (Trùng phân công)
ok, msg = validate_assignment_eligibility(teacher_toan, class_6A1, "toan", "2026-2027", current_assignments)
assert ok == False

print("✓ PASS: Toàn bộ kiểm thử phân công lớp & ràng buộc môn học thành công!")

## 3. Kiểm thử Gỡ Phân công (Soft Unassignment) & Lưu vết Lịch sử

In [ ]:
def soft_unassign(assignment_id, assignments, reason="Điều chuyển công tác"):
    now = datetime.utcnow().isoformat()
    updated = []
    for a in assignments:
        if a.get("id") == assignment_id or a.get("assignmentId") == assignment_id:
            new_a = dict(a)
            new_a["status"] = "INACTIVE"
            new_a["endDate"] = now
            new_a["reason"] = reason
            updated.append(new_a)
        else:
            updated.append(a)
    return updated

initial_assignments = [
    {"id": "asn_1", "teacherId": "TCH-2026-001", "classId": "class_6A1", "status": "ACTIVE"}
]

unassigned = soft_unassign("asn_1", initial_assignments)
assert len(unassigned) == 1, "Không được xóa vật lý bản ghi"
assert unassigned[0]["status"] == "INACTIVE"
assert "endDate" in unassigned[0]
print("✓ PASS: Xóa mềm phân công và bảo toàn lịch sử giảng dạy hoạt động chính xác!")